# Drexel University
## College of Computing and Informatics
## INFO 371: Data Mining Applications
## Assignment 4


## **DON'T FORGET TO PUT YOUR NAME BELOW**

## **NAME:** Hanzala Nadeem


### A. What to Hand In

1. A completed this Jupyter notebook.
2. Any data sets used in the assignment.
3. Any multimedia and images used in the assignment.

### B. How to Hand In

Submit your files through the course website in the Blackboard Learn system.

### C. When to Hand In

1. Submit your assignment no later than 11:59pm in the due date.
2. There will be a 10% (absolute value) deduction for each day of lateness, to a maximum of 3 days; assignments will not be accepted beyond that point. Missing work will earn a zero grade.

### D. Answer the questions.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import h5py

import keras

## Description
- In this assignment you are asked to build neural networks to classify butterflies.
- The dataset and problem are from Kaggle's Butterfly Image Classification: https://www.kaggle.com/datasets/phucthaiv02/butterfly-image-classification

## Use GPU in the Runtime type.
- Change the runtime type to the available GPU.
- Run the following code to check the availability of GPU.

In [ ]:
import tensorflow as tf

# Check for GPU availability
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

# Set TensorFlow to use GPU if available
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)  # Avoids memory allocation issues
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')  # Use only the first GPU
        print("Using GPU:", gpus[0])
    except RuntimeError as e:
        print(e)

## The Dataset
- The dataset contains 75 different classes of Butterflies.
- The dataset contains 6499 butterfly images with labels.
- The dataset is stored in the file `butterfly_labels.h5`.
- The following code loads the images as a $6499\times 64\times 64\times 3$ numpy array and the corresponding $6499$ labels.

In [ ]:
# copy the path of the file butterfly_labels.h5
input_h5_file= 'data/butterfly_labels.h5'

# load the saved HDF5 file
with h5py.File(input_h5_file, "r") as f:
    images = f["images"][:]  # Load images
    labels = f["labels"][:]  # Load labels (stored as bytes)

    # Convert bytes to strings
    labels = [label.decode("utf-8") for label in labels]

## Question 1 [5 points]: Show the labels of the first 10 images.

### Answer:

In [ ]:
for i in range(10):
  print(f"Image {i+1} label:", labels[i])

## Question 2 [5 points]: Show the shape of the `images` array.

### Answer:

In [ ]:
print("Shape of image array", images.shape)

## Question 3 [5 points]: Verify that there are 75 uniques labels and list the labels.

### Answer:

In [ ]:
unique_labels = np.unique(labels)
#get actual number of label
print("normal labels", len(labels))
print("unique labels", len(unique_labels))


## Question 4 [5 points]: Display the first 10 images in a $2\times 5$ grid. Label each image with its correponding label.

### Answer:

In [ ]:
for i in range(10):
  plt.subplot(2, 5, i+1)
  plt.imshow(images[i])
  plt.title(labels[i])
  plt.axis('off')
plt.show()

## Question 5 [5 points]: Show the range of pixel values in the `images` array as `[min, max]`.


### Answer:

In [ ]:
# Show the range of pixel values in the images array
min_pixel = images.min()
max_pixel = images.max()

print("Pixel value range:", [min_pixel, max_pixel])

## Question 6 [5 points]: Normalize the pixel values to the range of `[0, 1]`.

### Answer:

In [ ]:
# Normalize pixel values to the range [0, 1]
images = images / 255.0
print("New pixel range:", [images.min(), images.max()])

## Question 7 [10 points]: Convert the string labels to integer labels. Store the resultant integer labels in a numpy array `y`.

### Answer:

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Convert string labels to integer labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

# Convert to numpy array
y = np.array(y)

print("Integer labels:", y[:20])
print("Number of classes:", len(np.unique(y)))

## Quesetion 8 [10 points]: Split the normalized data and labels into training and test sets with the ratio 80:20. Ensure the label distribution is reflected in both training and test sets.

### Answer:

In [ ]:
from sklearn.model_selection import train_test_split
# Split the data into training and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    images,
    y, # Use the integer labels here
    test_size=0.2,
    stratify=y,   # ensures label distribution is preserved, using integer labels
    random_state=42
)

# Verify shapes
print("Training images shape:", X_train.shape)
print("Test images shape:", X_test.shape)
print("Training labels shape:", len(y_train))
print("Test labels shape:", len(y_test))

## Question 9 [10 points]: Write a Keras program to define a 2-layer neural network model.

The model should take images of shape (64, 64, 3) as input and classify them into 75 categories. It should consist of the following layers:

- An input layer matching the given image shape.
- A dense hidden layer with 512 neurons and ReLU activation.
- An output layer with 75 neurons and softmax activation for multi-class classification.
Print the model summary after defining the architecture.

### Answer:

In [ ]:
#import needed libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Input
# Define the model
model = Sequential([
    Input(shape=(64, 64, 3)),   # Input layer
    Flatten(),                  # Flatten image into a 1D vector
    Dense(512, activation='relu'),  # Hidden layer with 512 neurons
    Dense(75, activation='softmax') # Output layer for 75 classes
])

# Print the model summary
model.summary()

## Question 10 [10 points]: Compile, Train, and Test the Model:
- Compile the model using the Adam optimizer and sparse categorical cross-entropy as the loss function.
- Use `learning_rate=1e-3`.
- Use accuracy as the evaluation metric.
- Train the model on the provided training dataset (X_train, y_train) for 20 epochs with a batch size of 32.
- Use the test dataset (X_test, y_test) for validation.
- Store the training history in a variable.
- Plot the training and validation accuracy and loss over epochs after training.

### Answer:

In [ ]:
#import libraries
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
# Compile the model
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test)
)

# Plot Accuracy
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

## Question 11 [10 points]:  Do you find the performance of the 2-layer neural network satisfactory? In this question, design a neural network that outperforms the 2-layer NN.

### Answer:

In [ ]:
'''
The performance of the 2-layer neural network is moderate but not fully satisfactory.
While the model achieves around 28% validation accuracy, it is limited because it flattens the image into a one-dimensional vector before learning.
This approach ignores the spatial structure of images, such as edges, shapes, and textures,
which are important for image recognition tasks.

To improve performance, a Convolutional Neural Network (CNN) can be used. CNNs are
specifically designed for image data because they use convolutional filters to detect
important visual features while preserving spatial relationships between pixels.
'''

print(X_train.shape)

In [ ]:


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam

# Define CNN model
cnn_model = Sequential([
    Input(shape=(64,64,3)),

    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),

    Flatten(),

    Dense(128, activation='relu'),

    Dense(75, activation='softmax')
])

cnn_model.summary()
#compile and train model
cnn_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_cnn = cnn_model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test)
)
#draw visuals to compare results

plt.figure(figsize=(12,5))

# Accuracy plot
plt.subplot(1,2,1)
plt.plot(history_cnn.history['accuracy'], label='Training Accuracy')
plt.plot(history_cnn.history['val_accuracy'], label='Validation Accuracy')
plt.title('CNN Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss plot
plt.subplot(1,2,2)
plt.plot(history_cnn.history['loss'], label='Training Loss')
plt.plot(history_cnn.history['val_loss'], label='Validation Loss')
plt.title('CNN Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

## Question 12 [20 points]: Compare and analyze the performance of the 2 neural networks you have developed and evaluated.

- Compare the accuracy trends of both models. Which model achieves higher accuracy, and how quickly does it converge?
- Compare the training and validation loss curves. Does either model show signs of overfitting or underfitting?
- Explain how each model processes image data. Why does one model perform better than other for the butterfly classification?


Two neural network models were developed and evaluated: a 2-layer dense neural network and a convolutional neural network (CNN).



**Accuracy Comparison**

The 2-layer neural network achieved significantly higher accuracy compared to the CNN model. During training, the dense network showed a steady improvement in accuracy, reaching approximately 37% training accuracy and about 28% validation accuracy after 20 epochs. The model gradually improved as training progressed, indicating that it was able to learn meaningful patterns from the dataset.

In contrast, the CNN model achieved only about 2% accuracy on both the training and validation datasets, which is very close to random guessing for a classification problem with 75 classes (≈1.3%). The accuracy of the CNN stabilized very early in the training process and did not improve over the epochs, suggesting that the model failed to effectively learn useful features from the data.


**Training and Validation Loss Comparison**  


For the 2-layer neural network, both the training and validation loss curves steadily decreased throughout the training process. While the training loss decreased slightly faster than the validation loss, the two curves remained relatively close. This indicates that the model was learning effectively with only mild overfitting, which is common when the model begins to specialize on the training data.

For the CNN model, the training and validation loss curves decreased only slightly and remained almost flat across epochs. The loss values stayed relatively high, indicating that the model was not effectively minimizing the classification error. The nearly identical training and validation curves suggest underfitting, meaning the model failed to capture meaningful patterns in the data.

**How Each Model Processes Image Data**

The 2-layer neural network processes image data by first flattening the image into a one-dimensional vector before passing it through fully connected layers. This means the model treats the image as a long list of pixel values and does not explicitly consider spatial relationships between pixels.

The CNN model, on the other hand, processes images using convolutional filters that scan across the image to detect patterns such as edges, shapes, and textures. Convolutional layers preserve spatial relationships between neighboring pixels, making CNNs generally more effective for image classification tasks.

However, in this case the CNN did not perform well, likely due to factors such as insufficient model tuning, limited training data, or training configuration issues.

**Overall Conclusion**

Based on the evaluation results, the 2-layer dense neural network outperformed the CNN model for this experiment. It achieved higher accuracy, showed consistent improvement during training, and demonstrated effective learning of the dataset.

### Answer: